# GPT base model evaluation via OpenRouter

No GPU required — calls `openai/gpt-4o-mini` through OpenRouter.

## Run order

1. Put your OpenRouter key in the project `.env` file:
   `OPENROUTER_API_KEY=sk-or-v1-...` (or `OPENAI_API_KEY=...`)
2. Run the dependency cell, then restart the kernel if packages changed
3. Edit the **config cell** below, then run it
4. Run the client cell, then the helpers cell, then the evaluation cell

## Modifiable settings (config cell)

| Variable | Description |
|----------|-------------|
| `MODE` | `None` = all modes; or `"no_term"`, `"proper_term"`, `"random_term"` |
| `DATA_DIR` | Folder with JSONL files (e.g. `data/test`) |
| `OUTPUT_DIR` | Where predictions and metrics are written |
| `MAX_SAMPLES` | `None` = all samples; set an integer for a quick smoke test |
| `MODEL_NAME` | OpenRouter model id (default `openai/gpt-4o-mini`) |
| `TEMPERATURE` | Generation temperature (default `0`) |
| `MAX_TOKENS` | Max tokens per translation (default `256`) |

## Inputs / outputs

Inputs: one JSONL per language in `DATA_DIR`, e.g. `ende_dev_v1_test.jsonl`.

Outputs: `{OUTPUT_DIR}/*_{mode}_predictions.jsonl` and `{OUTPUT_DIR}/metrics_summary.json`.

In [1]:
%pip install -q openai sacrebleu tqdm python-dotenv
print("Restart kernel, then run from the config cell.")

Note: you may need to restart the kernel to use updated packages.
Restart kernel, then run from the config cell.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\vnpnk\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
from pathlib import Path

WORK_DIR = Path.cwd()

# --- user settings ---

MODE = "proper_term"  # None | "no_term" | "proper_term" | "random_term"
# DATA_DIR = "/dss/dsshome1/0E/go35bit2/Finetuning experiment/data/test"
# OUTPUT_DIR = "/dss/dsshome1/0E/go35bit2/Finetuning experiment/results/gpt_base"
DATA_DIR = "data/test"
OUTPUT_DIR = "results/gpt_base"
MAX_SAMPLES = None
MODEL_NAME = "openai/gpt-4o-mini"
TEMPERATURE = 0
MAX_TOKENS = 256

# --- resolved paths and modes ---

ALL_MODES = ["no_term", "proper_term", "random_term"]
LANG_GROUPS = ["ende", "enru", "enes"]

if MODE is None:
    MODES = ALL_MODES
else:
    if MODE not in ALL_MODES:
        raise ValueError(f"MODE must be None or one of {ALL_MODES}")
    MODES = [MODE]

DATA_ROOT = Path(DATA_DIR).expanduser().resolve()
if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f"DATA_DIR not found: {DATA_ROOT}")

OUTPUT_BASE = Path(OUTPUT_DIR).expanduser().resolve()

LANG_CONFIG = {
    "ende": {"ref_field": "de", "target_lang": "German", "output_tag": "de"},
    "enru": {"ref_field": "ru", "target_lang": "Russian", "output_tag": "ru"},
    "enes": {"ref_field": "es", "target_lang": "Spanish", "output_tag": "es"},
}

SAMPLE_SENTENCES: dict[str, list[dict[str, object]]] = {
    "ende": [
        {"en": "This service describes the deployed (run-time) state of SAP HANA database artifacts, for example: tables, views, or procedures, which have been created or adjusted by the SAP Integrated Development Environment (WebIDE) editors as a family of consistent design-time artifacts for all key SAP HANA platform database features.\n", "de": "Dieser Service beschreibt den implementierten Zustand (Laufzeitzustand) von SAP-HANA-Datenbankartefakten, z. B. Tabellen, Views oder Prozeduren, die von den SAP-Integrated-Development-Environment-Editoren (WebIDE-Editoren) als eine Familie konsistenter Entwurfszeit-Artefakte für alle wichtigen SAP-HANA-Plattform-Datenbankfunktionen erstellt oder angepasst wurden.\n", "proper_terms": {"design": "Entwurf", "state": "Zustand"}, "random_terms": {"artifacts": "Artefakten", "key": "wichtigen"}},
        {"en": "Request a checkup to perform a health diagnostic and better analyze what was wrong with your data flow.\n", "de": "Fordern Sie eine Kontrolle an, um eine Fehlerdiagnose durchzuführen und besser zu analysieren, was mit Ihrem Datenfluss falsch war.\n", "proper_terms": {"check": "Kontrolle"}, "random_terms": {"wrong": "falsch"}},
        {"en": "The data product is still available in the provider's data product list.\n", "de": "Das Datenprodukt ist weiterhin in der Datenproduktliste des Providers verfügbar.\n", "proper_terms": {"provider": "Provider"}, "random_terms": {"'s": "des"}}
    ],
    "enru": [
        {"en": "Indicates if a configuration item or configuration step is specific to a localized solution version.\n", "ru": "Указывает, являются ли позиция или шаг конфигурации специфичными для локализованной версии решения.\n", "proper_terms": {"item": "позиция"}, "random_terms": {"localized": "локализованной"}},
        {"en": "You run allocation cycles in the Run Allocations app.\n", "ru": "Для выполнения циклов перерасчета используется приложение Выполнить перерасчеты.\n", "proper_terms": {"run": "выполнить"}, "random_terms": {"cycles": "циклов"}},
        {"en": "Depending on your use case, you can choose between the following types of allocations:\n", "ru": "В зависимости от варианта использования можно выбрать один из следующих типов перерасчета:\n", "proper_terms": {"type": "тип"}, "random_terms": {"choose": "выбрать"}}
    ],
    "enes": [
        {"en": "In such cases you may use the Move Items or Merge feature.\n", "es": "En estos casos, puede utilizar la función Mover elementos o Fusionar .\n", "proper_terms": {"item": "elemento"}, "random_terms": {"Move": "Mover"}},
        {"en": "Save and Publish\n", "es": "Guardar y publicar\n", "proper_terms": {"save": "guardar"}, "random_terms": {"Publish": "publicar"}},
        {"en": "Required Permissions for SQL Server Trigger-Based Replication in the SAP HANA Smart Data Integration and SAP HANA Smart Data Quality Installation and Configuration Guide\n", "es": "Permisos necesarios para reproducción basada en desencadenador de SQL Server en la guía de instalación y configuración de Integración de datos inteligentes de SAP HANA y calidad de los datos inteligentes de SAP HANA\n", "proper_terms": {"replication": "reproducción"}, "random_terms": {"SQL": "SQL"}}
    ],
}


def discover_data_files(data_root: Path) -> dict[str, Path]:
    files: dict[str, Path] = {}
    for lang in LANG_GROUPS:
        matches = sorted(data_root.glob(f"{lang}_*.jsonl"))
        if not matches:
            raise FileNotFoundError(f"No JSONL for {lang} in {data_root}")
        if len(matches) > 1:
            raise ValueError(f"Multiple JSONL files for {lang}: {matches}")
        files[lang] = matches[0]
    return files


DATA_FILES = discover_data_files(DATA_ROOT)


def data_path(lang: str) -> Path:
    return DATA_FILES[lang]


def prediction_stem(lang: str) -> str:
    return DATA_FILES[lang].stem


print("KERNEL_CWD:", WORK_DIR)
print("DATA_DIR:", DATA_ROOT)
print("OUTPUT_DIR:", OUTPUT_BASE)
print("MODE:", MODE, "->", MODES)
print("MODEL_NAME:", MODEL_NAME)
print("TEMPERATURE:", TEMPERATURE)
print("MAX_TOKENS:", MAX_TOKENS)
print("MAX_SAMPLES:", "all" if MAX_SAMPLES is None else MAX_SAMPLES)
for lang in LANG_GROUPS:
    path = data_path(lang)
    print(f"  {lang}: {path.name} [ok] ({len(SAMPLE_SENTENCES[lang])} prompt examples)")

KERNEL_CWD: c:\Users\vnpnk\Documents\TUM\studies\semester 2\Terminology translation\terminology-translation\Finetuning experiment
DATA_DIR: C:\Users\vnpnk\Documents\TUM\studies\semester 2\Terminology translation\terminology-translation\Finetuning experiment\data\test
OUTPUT_DIR: C:\Users\vnpnk\Documents\TUM\studies\semester 2\Terminology translation\terminology-translation\Finetuning experiment\results\gpt_base
MODE: proper_term -> ['proper_term']
MODEL_NAME: openai/gpt-4o-mini
TEMPERATURE: 0
MAX_TOKENS: 256
MAX_SAMPLES: all
  ende: ende_dev_v1_test.jsonl [ok] (3 prompt examples)
  enru: enru_dev_v1_test.jsonl [ok] (3 prompt examples)
  enes: enes_dev_v1_test.jsonl [ok] (3 prompt examples)


In [3]:
import os
import time

from dotenv import load_dotenv
from openai import OpenAI

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
MAX_RETRIES = 3
RETRY_DELAY_S = 5


def find_env_file(start: Path) -> Path | None:
    for directory in [start, *start.parents]:
        env_path = directory / ".env"
        if env_path.exists():
            return env_path
    return None


env_file = find_env_file(WORK_DIR)
if env_file:
    load_dotenv(env_file)
    print("Loaded env from:", env_file)
else:
    print("No .env file found — using environment variables only")

API_KEY = (
    os.environ.get("OPENROUTER_API_KEY")
    or os.environ.get("OPENAI_API_KEY")
    or ""
).strip()
if not API_KEY:
    raise EnvironmentError(
        "No API key found. Set OPENROUTER_API_KEY or OPENAI_API_KEY in .env or your environment."
    )

if os.environ.get("OPENROUTER_MODEL"):
    MODEL_NAME = os.environ["OPENROUTER_MODEL"]

client = OpenAI(api_key=API_KEY, base_url=OPENROUTER_BASE_URL)
print("Provider: OpenRouter")
print("Base URL:", OPENROUTER_BASE_URL)
print("Model:", MODEL_NAME)
print("Temperature:", TEMPERATURE)
print("Max tokens:", MAX_TOKENS)
print("API key: set (not shown)")

Loaded env from: c:\Users\vnpnk\Documents\TUM\studies\semester 2\Terminology translation\terminology-translation\.env
Provider: OpenRouter
Base URL: https://openrouter.ai/api/v1
Model: openai/gpt-4o-mini
Temperature: 0
Max tokens: 256
API key: set (not shown)


In [4]:
import json
import re
from collections import Counter, defaultdict
from typing import Any

import sacrebleu
from tqdm import tqdm


def load_jsonl(path: Path, max_samples: int | None = None) -> list[dict[str, Any]]:
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            records.append(json.loads(line))
            if max_samples is not None and len(records) >= max_samples:
                break
    return records


def save_jsonl(path: Path, records: list[dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


def load_datasets() -> dict[str, list[dict[str, Any]]]:
    datasets = {}
    for lang in LANG_GROUPS:
        path = data_path(lang)
        datasets[lang] = load_jsonl(path, MAX_SAMPLES)
    return datasets


def terms_for_mode(sample: dict[str, Any], mode: str) -> dict[str, str]:
    if mode == "proper_term":
        return (sample.get("proper_terms") or {}).copy()
    if mode == "random_term":
        terms = (sample.get("random_terms") or {}).copy()
        for key in (sample.get("proper_terms") or {}):
            terms.pop(key, None)
        return terms
    return {}


def terminology_for_mode(sample: dict[str, Any], mode: str) -> dict[str, str] | None:
    terms = terms_for_mode(sample, mode)
    return terms or None


def strip_output_tags(text: str, output_tag: str) -> str:
    if not isinstance(text, str):
        return text
    return re.sub(rf"</?{re.escape(output_tag)}>", "", text, flags=re.IGNORECASE).strip()


def compute_bleu_chrf(hyps: list[str], refs: list[str]) -> dict[str, float]:
    bleu = sacrebleu.corpus_bleu(hyps, [refs])
    chrf = sacrebleu.corpus_chrf(hyps, [refs])
    return {"bleu": bleu.score, "chrf": chrf.score}


def _normalize_text(text: str) -> str:
    return " ".join(str(text).lower().split())


def _count_term_occurrences(text: str, term: str) -> int:
    text_norm = _normalize_text(text)
    term_norm = _normalize_text(term)
    return len(re.findall(r"\b" + re.escape(term_norm) + r"\b", text_norm))


def terminology_accuracy(preds: list[str], samples: list[dict[str, Any]], mode: str) -> dict[str, Any]:
    term_ratios: dict[str, float] = {}
    total_terms = 0
    for pred, sample in zip(preds, samples):
        source_text = sample.get("en", "")
        for src, tgt in terms_for_mode(sample, mode).items():
            total_terms += 1
            src_count = max(_count_term_occurrences(source_text, src), 1)
            tgt_count = _count_term_occurrences(pred, tgt)
            term_ratios[src] = min(tgt_count / src_count, 1.0)
    avg_ratio = sum(term_ratios.values()) / len(term_ratios) * 100 if term_ratios else None
    return {"total_terms": total_terms, "avg_ratio_pct": avg_ratio, "per_term_ratios": term_ratios}


def terminology_consistency(preds: list[str], samples: list[dict[str, Any]], mode: str) -> dict[str, Any]:
    term_to_candidates: dict[str, list[str]] = defaultdict(list)
    for pred, sample in zip(preds, samples):
        for src, tgt in terms_for_mode(sample, mode).items():
            candidate = tgt if str(tgt).lower() in str(pred).lower() else "<MISSING>"
            term_to_candidates[src].append(candidate)
    per_term = {}
    macro_scores = []
    weighted_scores = []
    for src, candidates in term_to_candidates.items():
        pseudo_ref = Counter(candidates).most_common(1)[0][0]
        matches = sum(1 for c in candidates if c == pseudo_ref)
        consistency = matches / len(candidates)
        per_term[src] = {
            "occ": len(candidates),
            "pseudo_ref": pseudo_ref,
            "matches": matches,
            "consistency": consistency,
        }
        macro_scores.append(consistency)
        weighted_scores.extend([consistency] * len(candidates))
    return {
        "per_term": per_term,
        "macro_avg_consistency": sum(macro_scores) / len(macro_scores) if macro_scores else None,
        "weighted_avg_consistency": sum(weighted_scores) / len(weighted_scores) if weighted_scores else None,
    }


def fmt_metric(value: float | None, digits: int = 2) -> str:
    return "N/A" if value is None else f"{value:.{digits}f}"


def format_terminology_block(terms: dict[str, str]) -> str:
    if not terms:
        return ""
    return "Terminology:\n" + "\n".join(f"{s} -> {t}" for s, t in terms.items()) + "\n"


def format_sample_examples(lang: str, mode: str) -> str:
    config = LANG_CONFIG[lang]
    ref_field = config["ref_field"]
    output_tag = config["output_tag"]
    blocks = []
    for i, example in enumerate(SAMPLE_SENTENCES[lang], 1):
        term_block = format_terminology_block(terms_for_mode(example, mode))
        ref = example.get(ref_field, "")
        blocks.append(
            f"Example {i}:\n"
            f"{term_block}"
            f"Input:\n<en> {example['en']} </en>\n"
            f"Output:\n<{output_tag}> {ref} </{output_tag}>"
        )
    return "Examples:\n\n" + "\n".join(blocks) + "\n\n"


def chat_completion_with_retry(messages: list[dict[str, str]]) -> str:
    last_error: Exception | None = None
    for attempt in range(MAX_RETRIES):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=messages,
                temperature=TEMPERATURE,
                max_tokens=MAX_TOKENS,
            )
            return response.choices[0].message.content.strip()
        except Exception as exc:
            last_error = exc
            if attempt < MAX_RETRIES - 1:
                time.sleep(RETRY_DELAY_S * (2 ** attempt))
    raise RuntimeError(f"OpenRouter API failed after {MAX_RETRIES} attempts") from last_error


def translate_sample(
    sample_en: str,
    terminology: dict[str, str] | None,
    target_lang: str,
    output_tag: str,
    lang: str,
    mode: str,
) -> str:
    examples_block = format_sample_examples(lang, mode)
    term_block = format_terminology_block(terminology or {})
    if term_block:
        term_block += "\n"
    prompt = f"""You are a translation assistant.

Translate the English text to {target_lang}.

Rules:
1. Output only in this format: <{output_tag}> ... </{output_tag}>
2. Use the terminology mappings exactly as provided.
3. Do not explain anything.
4. Translate only from English to {target_lang}.

{examples_block}{term_block}Input:
<en> {sample_en} </en>
"""
    messages = [
        {"role": "system", "content": "You are a helpful translation assistant."},
        {"role": "user", "content": prompt},
    ]
    return chat_completion_with_retry(messages)

In [ ]:
def prediction_filename(lang: str, mode: str) -> str:
    return f"{prediction_stem(lang)}_{mode}_predictions.jsonl"


def run_mode(
    lang: str,
    mode: str,
    samples: list[dict[str, Any]],
    output_dir: Path,
    config: dict[str, str],
) -> dict[str, Any]:
    ref_field = config["ref_field"]
    preds = []
    records = []
    for sample in tqdm(samples, desc=f"{lang}/{mode}"):
        pred = translate_sample(
            sample.get("en", ""),
            terminology_for_mode(sample, mode),
            config["target_lang"],
            config["output_tag"],
            lang,
            mode,
        )
        preds.append(pred)
        record = sample.copy()
        record[f"prediction_{mode}"] = pred
        record[f"prediction_{mode}_clean"] = strip_output_tags(pred, config["output_tag"])
        records.append(record)
    clean_preds = [strip_output_tags(p, config["output_tag"]) for p in preds]
    pred_path = output_dir / prediction_filename(lang, mode)
    save_jsonl(pred_path, records)
    metrics: dict[str, Any] = {}
    if samples and ref_field in samples[0]:
        refs = [sample.get(ref_field, "") for sample in samples]
        metrics.update(compute_bleu_chrf(clean_preds, refs))
        term_acc = terminology_accuracy(clean_preds, samples, mode)
        term_cons = terminology_consistency(clean_preds, samples, mode)
        metrics["terminology_accuracy"] = term_acc
        metrics["terminology_consistency"] = term_cons
        print(
            f"[{lang}/{mode}] BLEU={fmt_metric(metrics['bleu'])} "
            f"chrF={fmt_metric(metrics['chrf'])} "
            f"term_acc={fmt_metric(term_acc['avg_ratio_pct'])}% "
            f"macro_cons={fmt_metric(term_cons['macro_avg_consistency'])} "
            f"weighted_cons={fmt_metric(term_cons['weighted_avg_consistency'])}"
        )
    else:
        print(f"[{lang}/{mode}] no reference field '{ref_field}' — metrics skipped")
    return {"predictions_file": str(pred_path), "metrics": metrics}


datasets = load_datasets()
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)

summary = {
    "data_dir": str(DATA_ROOT),
    "output_dir": str(OUTPUT_BASE),
    "mode": MODE,
    "modes_run": MODES,
    "prompt_examples_per_lang": {lang: len(SAMPLE_SENTENCES[lang]) for lang in LANG_GROUPS},
    "provider": "openrouter",
    "base_url": OPENROUTER_BASE_URL,
    "model": MODEL_NAME,
    "temperature": TEMPERATURE,
    "max_tokens": MAX_TOKENS,
    "max_samples": MAX_SAMPLES,
    "languages": {},
}

for lang in LANG_GROUPS:
    config = LANG_CONFIG[lang]
    samples = datasets[lang]
    print(f"\n=== {lang}: {len(samples)} samples → {config['target_lang']} ===")
    lang_results = {mode: run_mode(lang, mode, samples, OUTPUT_BASE, config) for mode in MODES}
    summary["languages"][lang] = {
        "data_file": str(data_path(lang)),
        "sample_count": len(samples),
        **config,
        "modes": lang_results,
    }

metrics_path = OUTPUT_BASE / "metrics_summary.json"
with metrics_path.open("w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("\nDone.", metrics_path)